## Matrix Vecotr Broadcasting 

In [2]:
import torch 
import numpy as np

In [3]:
## Matrix dimensions
 

D = 3
n_particles = 2**15

dtype = torch.float32
device_gpu = 'cuda'


N_REPEATS_GPU = 2*12

''' Initialization of position matrices  '''
q_DxN = torch.randn(D, n_particles, device=device_gpu, dtype=dtype) 
q1_Dx1 = q_DxN[:,0]
delta = torch.empty_like(q_DxN)
abs_1xN = torch.empty(1, n_particles, device=device_gpu, dtype=dtype) 

print(q1_Dx1)

delta = q_DxN - q1_Dx1[:, None]
sq_norm_1xN = ((q_DxN - q1_Dx1[:, None])**2).sum(dim=0)
#sq_norm_1xN_torch = (q_DxN - q1_Dx1[:, None]).sqnorm2()


print(delta)
print(sq_norm_1xN)



tensor([-1.0316,  2.2117, -0.9415], device='cuda:0')
tensor([[ 0.0000,  1.1721,  2.1733,  ..., -0.3017,  1.6260,  0.8275],
        [ 0.0000, -3.9385, -1.5415,  ..., -1.9280, -4.7204, -1.5915],
        [ 0.0000,  1.0149,  2.1799,  ...,  2.1976,  0.4634,  0.5304]],
       device='cuda:0')
tensor([ 0.0000, 17.9159, 11.8517,  ...,  8.6376, 25.1406,  3.4991],
       device='cuda:0')


In [6]:
import torch
import math

D = 3
n_particles = 2**15
dtype = torch.float32
device_gpu = 'cuda' if torch.cuda.is_available() else 'cpu'

# Dummy Parameter für das Potenzial
r0 = 1.0
c = 10.0
r0_2 = r0**2

# --- Initialisierung ---
# WICHTIG: Deine Dimensionen waren (D, N). 
# PyTorch Standard ist oft (N, D), aber wir bleiben bei deinem (D, N) Layout.
q_DxN = torch.randn(D, n_particles, device=device_gpu, dtype=dtype) 

# Speicher für die Ergebnisse (Kräfte und Potenzial)
total_forces_DxN = torch.zeros(D, n_particles, device=device_gpu, dtype=dtype)
total_potential = 0.0

print(f"Berechnung für {n_particles} Teilchen startet...")

# --- Die Schleife (Der Kern des Algorithmus) ---
# Wir iterieren über jedes Teilchen i. 
# Teilchen i ist der "Beobachter", der die Kraft von allen j spürt.

for i in range(n_particles):
    # 1. Position des aktuellen Teilchens i holen
    # Shape: (D,) -> wir machen (D, 1) draus für Broadcasting gegen (D, N)
    q_i = q_DxN[:, i].unsqueeze(1) 
    
    # 2. Differenzvektor zu ALLEN anderen Teilchen berechnen
    # q_DxN ist (D, N), q_i ist (D, 1). 
    # Ergebnis delta ist (D, N). Das ist der Vektor r_j - r_i
    # Achtung Vorzeichen: Für Kraft auf i brauchen wir (r_i - r_j) oder (r_j - r_i)
    # Hier: diff = r_j - r_i.  Abstand ist symmetrisch.
    diff = q_DxN - q_i 
    
    # 3. Quadratischer Abstand berechnen
    # Wir quadrieren elementweise und summieren über die Dimension 0 (die Koordinaten x,y,z)
    # Ergebnis ist (N,)
    r_sq_N = (diff**2).sum(dim=0)
    
    # --- Optimierung: Selbstinteraktion ignorieren ---
    # r_sq für i=i ist 0. Das würde zu Division durch Null oder falschen Kräften führen.
    # Wir setzen den Abstand für sich selbst auf Unendlich oder filtern später.
    # Trick: Wir addieren eine "Maske" oder setzen r_sq[i] = inf
    # (Aber bei Gauss e^-r stört 0 abstand nicht bei der Kraft (Kraft ist 0), 
    # nur beim Potential (Potential ist c))
    
    # 4. Gauß-Term berechnen
    # exp(-r^2 / 2r0^2)
    exp_term = torch.exp(-r_sq_N / (2 * r0_2))
    
    # 5. Kraftbeitrag auf Teilchen i berechnen
    # Kraft Formel: F_ij = (c / r0^2) * exp(...) * (r_i - r_j)
    # Unser 'diff' ist (r_j - r_i) = -(r_i - r_j).
    # Also Kraft auf i zeigt in Richtung diff (bei abstoßung).
    # F_i = sum_j ( Skalar * diff_j )
    
    force_magnitude = exp_term * (c / r0_2) # Shape (N,)
    
    # Jetzt Vektor berechnen: Skalar * Vektor. 
    # diff ist (D, N), force_magnitude ist (N,). Broadcasting passt.
    force_vecs = diff * force_magnitude.unsqueeze(0) # Shape (D, N)
    
    # Alles aufsummieren zu einer Gesamtkraft auf Teilchen i
    force_on_i = force_vecs.sum(dim=1) # Summe über alle j -> Ergebnis (D,)
    
    # Speichern
    total_forces_DxN[:, i] = force_on_i
    
    # 6. Potenzialbeitrag (optional hier akkumulieren)
    # E_i = 0.5 * sum( c * exp )
    # Wir summieren hier alles auf und korrigieren am Ende die Selbstinteraktion und Doppelzählung
    total_potential += (c * exp_term).sum()

# --- Nachbearbeitung ---

# Korrektur des Potenzials
# 1. Selbstinteraktion abziehen (i=j): Beitrag war c * exp(0) = c. Das passierte N mal.
total_potential -= n_particles * c

# 2. Doppelzählung korrigieren (Paar i-j und j-i wurde jeweils addiert)
total_potential *= 0.5

print("Fertig.")
print("Form der Kräfte:", total_forces_DxN.shape)
print("Gesamtpotenzial:", total_potential.item())

Berechnung für 32768 Teilchen startet...
Fertig.
Form der Kräfte: torch.Size([3, 32768])
Gesamtpotenzial: 1029186112.0


In [18]:
import torch

# --- Konfiguration ---
D = 3
n_particles = 2**17  # 32768
dtype = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

r0 = 1.0
c = 10.0
r0_2 = r0**2

# --- Setup ---
# Wir nutzen hier (N, D) Layout, das ist für PyTorch Broadcasting oft intuitiver
# Wir transponieren dein (D, N) also kurz für die Rechnung.
q_DxN = torch.randn(D, n_particles, device=device, dtype=dtype)
positions = q_DxN.T  # Shape (N, 3)

forces = torch.zeros_like(positions)
total_potential = 0.0

# WICHTIG: Die Blockgröße (Batch Size)
# 1024 oder 2048 ist meist ein "Sweet Spot" für GPUs.
# Die Zwischenmatrix wird (BATCH_SIZE, N, 3) groß sein.
# Bei 1024 * 32768 * 3 * 4 Bytes sind das ca. 400 MB VRAM. Das passt locker!
BATCH_SIZE = 2**11

print(f"Starte Berechnung mit Batch-Size {BATCH_SIZE}...")

# --- Die Batch-Schleife ---
for i in range(0, n_particles, BATCH_SIZE):
    # 1. Definiere den aktuellen Block (Chunk) von Teilchen i
    # Dies sind die "Empfänger" der Kraft
    q_chunk = positions[i : i + BATCH_SIZE]  # Shape: (B, 3)
    
    # 2. Differenzmatrix berechnen
    # q_chunk: (B, 1, 3)
    # positions: (1, N, 3) (Alle anderen sind die "Sender")
    # diff: (B, N, 3) -> Hier nutzen wir Broadcasting, aber nur für den Chunk!
    diff = q_chunk[:, None, :] - positions[None, :, :]
    
    # 3. Abstandsquadrate
    # Ergebnis Shape: (B, N)
    r_sq = (diff ** 2).sum(dim=-1)
    
    # 4. Kräfte und Potenzial berechnen
    exp_term = torch.exp(-r_sq / (2 * r0_2)) # Shape (B, N)
    
    # Potenzial für diesen Chunk aufsummieren
    total_potential += (c * exp_term).sum()
    
    # Kraft: F_ij Vektoren
    forces_magnitude = exp_term * (c / r0_2) # Shape (B, N)
    
    # Vektoren gewichten: (B, N, 1) * (B, N, 3) -> (B, N, 3)
    force_vecs = forces_magnitude[..., None] * diff
    
    # 5. Summe über alle j (Dimension 1), um Gesamtkraft auf die i's zu kriegen
    # Ergebnis Shape: (B, 3)
    forces_chunk = force_vecs.sum(dim=1)
    
    # 6. Ins große Array schreiben
    forces[i : i + BATCH_SIZE] = forces_chunk

# --- Nachbearbeitung ---
total_potential -= n_particles * c  # Selbstinteraktion abziehen
total_potential *= 0.5              # Doppelzählung korrigieren

# Zurück ins (D, N) Format transponieren, falls nötig
forces_DxN = forces.T 

print("Fertig.")
print("Forces Shape:", forces_DxN.shape)
print("Potential:", total_potential.item())

Starte Berechnung mit Batch-Size 2048...
Fertig.
Forces Shape: torch.Size([3, 131072])
Potential: 16532346880.0


In [12]:
import torch

# --- Konfiguration ---
D = 3
n_particles = 2**18  # 32768
dtype = torch.float32
device_gpu = 'cuda' if torch.cuda.is_available() else 'cpu'

# Dummy Parameter
r0 = 1.0
c = 10.0
r0_2 = r0**2

# --- Initialisierung ---
# Dein Layout: (D, N) -> (3, 32768)
q_DxN = torch.randn(D, n_particles, device=device_gpu, dtype=dtype) 

# Speicher für Ergebnisse
total_forces_DxN = torch.empty((D, n_particles), device=device_gpu, dtype=dtype)
total_potential = 0.0

# BATCH_SIZE: Wie viele Teilchen 'i' bearbeiten wir gleichzeitig?
# 512 bis 4096 ist meistens optimal für GPUs.
# Je größer, desto schneller, aber desto mehr VRAM wird benötigt.
BATCH_SIZE = 2*12 

print(f"Starte parallele Berechnung mit Batch-Size {BATCH_SIZE}...")

# --- Die Batch-Schleife ---
# Wir springen in Schritten von BATCH_SIZE durch die Teilchen
for i_start in range(0, n_particles, BATCH_SIZE):
    i_end = min(i_start + BATCH_SIZE, n_particles)
    
    # 1. Hole einen BLOCK von Teilchen "i"
    # Shape q_chunk: (D, Batch_Size)  z.B. (3, 1024)
    q_chunk = q_DxN[:, i_start:i_end]
    
    # 2. Differenzvektor berechnen (Broadcasting)
    # Wir wollen Matrix: (D, Batch_Size, N_Particles)
    # q_chunk:       (D, Batch, 1)  <- unsqueeze(-1)
    # q_DxN (alle):  (D, 1, N)      <- unsqueeze(1)
    # diff:          (D, Batch, N)
    diff = q_chunk.unsqueeze(-1) - q_DxN.unsqueeze(1) # Achtung: Vorzeichenkonvention prüfen (hier r_i - r_j)
    
    # Anmerkung zum Vorzeichen: Im Single-Loop hattest du q_all - q_i (also r_j - r_i).
    # Hier rechnen wir q_chunk - q_all (r_i - r_j). Das ist genau das Negative.
    # Da (diff^2) das Vorzeichen egalisiert und wir später diff * scalar rechnen, 
    # müssen wir nur aufpassen, in welche Richtung die Kraft zeigt.
    # Abstoßende Kraft auf i zeigt in Richtung (r_i - r_j). Also passt das hier.

    # 3. Quadratischer Abstand
    # Summe über Dimension 0 (x,y,z).
    # Ergebnis Shape: (Batch, N)
    r_sq_Batch_N = (diff**2).sum(dim=0)

    # 4. Gauß-Term (Elementweise auf der (Batch, N) Matrix)
    exp_term = torch.exp(-r_sq_Batch_N / (2 * r0_2)) # Shape (Batch, N)
    
    # 5. Kraftbeitrag (Vektorisierung)
    # force_magnitude: (Batch, N)
    force_magnitude = exp_term * (c / r0_2)
    
    # diff hat Shape (D, Batch, N)
    # force_magnitude hat Shape (Batch, N) -> muss zu (1, Batch, N) für Broadcasting
    force_vecs = diff * force_magnitude.unsqueeze(0) 
    
    # 6. Summation über alle Partner j (Dimension 2: N_Particles)
    # Wir wollen pro Teilchen im Batch einen Kraftvektor (D, 1)
    # Ergebnis Shape: (D, Batch)
    forces_chunk = force_vecs.sum(dim=2)
    
    # In das große Array speichern
    total_forces_DxN[:, i_start:i_end] = forces_chunk
    
    # 7. Potenzial aufsummieren
    # Hier summieren wir einfach alle Werte im Block auf
    total_potential += (c * exp_term).sum()

# --- Nachbearbeitung ---
# Identisch zu vorher
total_potential -= n_particles * c  # Selbstinteraktion (Diagonale)
total_potential *= 0.5              # Doppelzählung

print("Fertig.")
print("Form der Kräfte:", total_forces_DxN.shape)
print("Gesamtpotenzial:", total_potential.item())

Starte parallele Berechnung mit Batch-Size 24...
Fertig.
Form der Kräfte: torch.Size([3, 262144])
Gesamtpotenzial: 66000965632.0
